In [1]:
%cd '/home/mojtaba.kananisarche/'

/home/mojtaba.kananisarche


In [2]:
from Zh.src.utils.common import timeit
from Zh.src.utils.load_data import load_data, _extract_headers

In [3]:
file_path = '/work/long_lab/Mojtaba/data/UTA/1.3.3-augmented_embedded_preprocessed_normalized/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv'
file_data = load_data(file_path)
groupby_column, sortby_column, target_column = "file_name", "frame", "target"
columns_to_drop = [
    "file_name",
    "file_name_1",
    "type",
    "frame",
    "fold",
    "subject_id",
]
column_names = []
column_names = column_names or _extract_headers(file_path)
feature_columns = [
    column_name
    for column_name in column_names
    if column_name not in 
    columns_to_drop + [target_column, 
                       sortby_column, 
                       groupby_column]
]
slice_size = 4000
step_size = 4000
discard_threshold_ratio = 0.7

2025-05-03 14:43:18,090 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.3-augmented_embedded_preprocessed_normalized/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18070, 135)


Function load_data Took 1.0964 seconds


In [ ]:
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional

@timeit(verbose=0)
def reshape_data_rnn(
    df: pd.DataFrame,
    feature_columns: List[str],
    slice_size: int,
    step_size: Optional[int] = None,
    groupby_column: str = "file_name",
    sortby_column: str = "frame",
    target_column: str = "target",
    discard_threshold_ratio: float = 0.6,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Reshape data for RNN input by grouping, slicing, and padding sequences - optimized version.

    Args:
        df (pd.DataFrame)              : Input dataframe.
        feature_columns (List[str])    : List of feature column names.
        slice_size (int)               : Maximum number of slices in each chunk.
        step_size (int)                : Step size for moving to the next chunk.
        groupby_column (str)           : Column to group by (e.g., video or file name).
        sortby_column (str)            : Column to sort within each group (e.g., frame index).
        target_column (str)            : Column containing target labels.
        discard_threshold_ratio (float): Last slice in each video will be discarded if the number of its frames is less than
                                         discard_threshold_ratio*slice_size and will be post padded with zero and kept otherwise.
    Returns:
        Tuple[np.ndarray, np.ndarray, np.ndarray]: Feature sequences (X), corresponding labels (y), and group identifiers (g).
    """
    step_size = step_size or slice_size
    discard_threshold_frame = int(discard_threshold_ratio * slice_size)
    
    # Pre-allocate lists with estimated size to avoid resizing
    total_groups = len(df[groupby_column].unique())
    estimated_total_chunks = max(1, df.shape[0] // step_size) + total_groups #considering all last chunks in each group is higher than threshold

    print(estimated_total_chunks)
    
    # Pre-allocate arrays
    feature_count = len(feature_columns)
    all_chunks = np.zeros((estimated_total_chunks, slice_size, feature_count))
    all_labels = np.empty(estimated_total_chunks, dtype=df[target_column].dtype)
    all_groups = np.empty(estimated_total_chunks, dtype=df[groupby_column].dtype)
    
    chunk_idx = 0
    
    # Process groups in parallel if possible
    for group_name, group_df in df.groupby(groupby_column):
        # Sort efficiently
        group_df = group_df.sort_values(by=sortby_column)
        
        # Get the target value for this group
        y_group = group_df[target_column].iloc[0]  # Faster than using unique()
        
        # Extract features as a numpy array - much faster than .values.tolist()
        features = group_df[feature_columns].to_numpy()
        total_frames = len(features)
        
        # Process all full slices
        start_idx = 0

        for start_idx in range(0, total_frames - slice_size + 1, step_size):
            
            # Store chunk by view, not copy
            all_chunks[chunk_idx] = features[start_idx:start_idx + slice_size]
            all_labels[chunk_idx] = y_group
            all_groups[chunk_idx] = group_name
            chunk_idx += 1
        
        # Process the last partial slice if needed
        remaining_frames = total_frames - (start_idx + slice_size)
        if remaining_frames > discard_threshold_frame:
            
            all_chunks[chunk_idx, :remaining_frames] = features[start_idx + slice_size:]
            all_labels[chunk_idx] = y_group
            all_groups[chunk_idx] = group_name
            chunk_idx += 1
    
    # Trim arrays to actual size
    all_labels = all_labels[:chunk_idx]
    all_groups = all_groups[:chunk_idx]
    all_chunks = all_chunks[:chunk_idx]
    
    return all_chunks, all_labels, all_groups

In [209]:
from keras.src.utils import pad_sequences

@timeit(verbose=0)
def reshape_data_rnn(
    df: pd.DataFrame,
    feature_columns: List[str],
    slice_size: int,
    step_size: Optional[int],
    groupby_column: str = "file_name",
    sortby_column: str = "frame",
    target_column: str = "target",
    discard_threshold_ratio: float = 0.6,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Reshape data for RNN input by grouping, slicing, and padding sequences.

    Args:
        df (pd.DataFrame)              : Input dataframe.
        feature_columns (List[str])    : List of feature column names.
        slice_size (int)               : Maximum number of slices in each chunk.
        step_size (int)                : Step size for moving to the next chunk.
        groupby_column (str)           : Column to group by (e.g., video or file name).
        sortby_column (str)            : Column to sort within each group (e.g., frame index).
        target_column (str)            : Column containing target labels.
        discard_threshold_ratio (float): Last slice in each video will be discarded if the number of its frames is less than
                                         discard_threshold_ratio*max_slice and will be post padded with zero and kept otherwise.
    Returns:
        Tuple[np.ndarray, np.ndarray]: Feature sequences (X) and corresponding labels (y).
    """
    step_size = step_size or slice_size
    X, y, g = [], [], []

    for group_identifier, group in df.groupby(groupby_column):
        # Sort by the specified column to maintain temporal order
        group = group.sort_values(by=sortby_column, ascending=True)

        y_group = group[target_column].unique()
        assert (
            len(y_group) == 1
        ), f"Expected one unique target value, but found {len(y_group)}"
        y_group = y_group[0]
        assert (target_column not in feature_columns), "Target Column Appeared in the features"
        assert (groupby_column not in feature_columns), "GroupBy Column Appeared in the features"
        # Extract features as a list of lists
        slices = group[feature_columns].values.tolist()
        # Create chunks with the specified step size

        start_idx = 0
        for start_idx in range(0, len(slices) - slice_size + 1, step_size):
            chunk = slices[start_idx : start_idx + slice_size]
            X.append(chunk)
            y.append(y_group)
            g.append(group_identifier)

        discard_threshold_frame = int(discard_threshold_ratio * slice_size)
        if len(slices) - (start_idx + slice_size) > discard_threshold_frame:
            chunk = slices[start_idx + slice_size : len(slices)]
            X.append(chunk)
            y.append(y_group)
            g.append(group_identifier)

    # Pad sequences to ensure uniform length
    X_padded = pad_sequences(X, maxlen=slice_size, padding="post", dtype="float32")

    return np.array(X_padded), np.array(y), np.array(g)

In [210]:
import json

In [211]:
file_path = '/work/long_lab/Mojtaba/data/UTA/1.3.3-augmented_embedded_preprocessed_normalized/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv'
file_data = load_data(file_path)
groupby_column, sortby_column, target_column = "file_name", "frame", "target"
columns_to_drop = [
    "file_name",
    "file_name_1",
    "type",
    "frame",
    "fold",
    "subject_id",
]
column_names = []
column_names = column_names or _extract_headers(file_path)
feature_columns = [
    column_name
    for column_name in column_names
    if column_name not in 
    columns_to_drop + [target_column, 
                       sortby_column, 
                       groupby_column]
]
slice_size = 4000
step_size = 4000
discard_threshold_ratio = 0

2025-05-03 16:08:24,441 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.3-augmented_embedded_preprocessed_normalized/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18070, 135)


Function load_data Took 1.0103 seconds


In [212]:
file_path = '/work/long_lab/Mojtaba/data/UTA/1.3.4-augmented_embedded_preprocessed_normalized_npy/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.npy'
file_data_temp = load_data(file_path)

2025-05-03 16:08:24,507 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.4-augmented_embedded_preprocessed_normalized_npy/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.npy, data shape: (18070, 133)


Function load_data Took 0.0431 seconds


In [128]:
X1, y1, g1 = reshape_data_rnn_optimized(df                      = file_data, 
                                        feature_columns         = feature_columns,
                                        slice_size              = slice_size,
                                        step_size               = step_size,
                                        groupby_column          = groupby_column,
                                        sortby_column           = sortby_column,
                                        target_column           = target_column,
                                        discard_threshold_ratio = discard_threshold_ratio)

5
Function reshape_data_rnn_optimized Took 0.0295 seconds


In [129]:
X1.shape

(4, 4000, 128)

In [213]:
X1, y1, g1 = reshape_data_rnn_optimized(df                      = file_data, 
                                        feature_columns         = feature_columns,
                                        slice_size              = slice_size,
                                        step_size               = step_size,
                                        groupby_column          = groupby_column,
                                        sortby_column           = sortby_column,
                                        target_column           = target_column,
                                        discard_threshold_ratio = discard_threshold_ratio)

5
2070
Function reshape_data_rnn_optimized Took 0.0396 seconds


In [214]:
X1.shape

(5, 4000, 128)

In [221]:
X1[-1][2068][0]

np.float64(0.6599489937607877)

In [224]:
X1[-1][2069][0]

np.float64(0.7096496660823832)

In [139]:
X2, y2, g2 = reshape_data_rnn(df                      = file_data, 
                              feature_columns         = feature_columns,
                              slice_size              = slice_size,
                              step_size               = step_size,
                              groupby_column          = groupby_column,
                              sortby_column           = sortby_column,
                              target_column           = target_column,
                              discard_threshold_ratio = discard_threshold_ratio)

Function reshape_data_rnn Took 0.6771 seconds


In [49]:
g2

array(['1_01_0_30fps_brightness_alpha0.8_beta-10.csv',
       '1_01_0_30fps_brightness_alpha0.8_beta-10.csv',
       '1_01_0_30fps_brightness_alpha0.8_beta-10.csv',
       '1_01_0_30fps_brightness_alpha0.8_beta-10.csv'], dtype='<U44')

In [140]:
a = np.isclose(X1, X2)

In [144]:
sum(sum(a))

array([16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000, 16000,
       16000, 16000]

In [32]:
X2[0][0].astype(float)

array([ 0.65185821,  0.19688255, -0.03148719, -0.17134865,  0.52859354,
        0.72274071, -0.70296073, -0.27598718,  0.76629716,  0.77871662,
        0.34688005, -0.06536248, -0.33076945, -0.49611551, -0.01474306,
       -0.05188863,  1.20419252, -0.72340596, -0.35993081, -0.70616388,
        0.05836924, -0.18616647,  1.04749894,  0.31779331,  0.14596085,
        0.08725449,  0.48383507, -0.0165069 ,  1.11340475,  0.78685313,
       -0.29265213, -0.17102125, -1.01625264, -0.59360528,  1.81624663,
        0.51287818, -1.90553606,  0.13143919, -0.73062706,  1.29558194,
       -0.59514874, -2.07187819,  0.83396536,  2.46221447,  1.29766226,
       -1.2097348 ,  0.24449366, -0.40390459,  0.21223766, -0.83421713,
        0.52372879, -0.32449889, -1.24573541,  0.29226649, -0.96584135,
        0.06099323,  0.21635135, -0.12416711, -2.67957759, -1.68777597,
       -2.02655554, -1.47316086,  1.00164628, -3.05148363, -1.87167799,
        0.75814974, -0.77867728,  0.60303533,  0.22161138, -1.33

In [9]:
file_path_1 = '/work/long_lab/Mojtaba/data/UTA/1.3.3-augmented_embedded_preprocessed_normalized/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv'

@timeit
def df_read(file_path, k=1):
    for i in range(k-1):
        pd.read_csv(file_path)
    return pd.read_csv(file_path)
df = df_read(file_path_1)

npy = df.select_dtypes('number').to_numpy()
np.save('/home/mojtaba.kananisarche/Zh/temp/temp.npy', npy)

@timeit
def np_read(file_path, k=1):
    for i in range(k-1):
        np.load(file_path)
    return np.load(file_path)

file_path_2 = '/home/mojtaba.kananisarche/Zh/temp/temp.npy'

dfo = df_read(file_path_1, 100)
npo = np_read(file_path_2, 100)

Function df_read('/work/long_lab/Mojtaba/data/UTA/1.3.3-augmented_embedded_preprocessed_normalized/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',) {} Took 1.0155 seconds
Function df_read('/work/long_lab/Mojtaba/data/UTA/1.3.3-augmented_embedded_preprocessed_normalized/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv', 100) {} Took 101.0020 seconds
Function np_read('/home/mojtaba.kananisarche/Zh/temp/temp.npy', 100) {} Took 0.9553 seconds


In [6]:
import json

In [7]:
with open('/work/long_lab/Mojtaba/data/UTA/1.3.4-augmented_embedded_preprocessed_normalized_npy/columns.json', 'r') as json_file:
    j = json.load(json_file)

In [23]:
@timeit
def do():
    pd.DataFrame(np.load('/work/long_lab/Mojtaba/data/UTA/1.3.4-augmented_embedded_preprocessed_normalized_npy/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.npy'), columns=j)
do()

Function do() {} Took 0.0101 seconds


In [24]:
@timeit
def do():
    np.load('/work/long_lab/Mojtaba/data/UTA/1.3.4-augmented_embedded_preprocessed_normalized_npy/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.npy')
do()

Function do() {} Took 0.0094 seconds


Function df_read('/work/long_lab/Mojtaba/data/UTA/1.3.3-augmented_embedded_preprocessed_normalized/0/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv', 100) {} Took 99.2151 seconds
Function np_read('/home/mojtaba.kananisarche/Zh/temp/temp.npy', 100) {} Took 0.9666 seconds


In [58]:
npo

array([[ 0.00000000e+00,  1.00000000e+00,  1.00000000e+00, ...,
         4.95046897e-01, -1.63141547e+00, -1.88382430e-02],
       [ 1.00000000e+00,  1.00000000e+00,  1.00000000e+00, ...,
         3.50112256e-01, -1.97121260e+00, -5.67894179e-02],
       [ 2.00000000e+00,  1.00000000e+00,  1.00000000e+00, ...,
         3.43622645e-01, -1.52223476e+00, -1.19732830e-01],
       ...,
       [ 1.80670000e+04,  1.00000000e+00,  1.00000000e+00, ...,
         1.21894275e-01, -2.11715825e+00,  5.30065335e-01],
       [ 1.80680000e+04,  1.00000000e+00,  1.00000000e+00, ...,
         7.54872919e-04, -2.03917202e+00,  6.10595876e-01],
       [ 1.80690000e+04,  1.00000000e+00,  1.00000000e+00, ...,
         1.04892892e-02, -2.12941323e+00,  6.32811198e-01]])

In [46]:
npo = np_read(file_path_2)

Function np_read('/home/mojtaba.kananisarche/Zh/temp/temp.npy',) {} Took 0.4291 seconds


In [3]:
from typing import Optional, List, Dict
import pandas as pd
from src.utils.file_folder import get_file_paths
import pandas as pd
import os

def _get_train_val_test_file_paths(
    input_path: str,
    primary_column_name: str,
    secondary_column_name: Optional[str] = None,
) -> List[Dict]:

    file_paths = get_file_paths(folder_path=input_path, depth=2, EXT_LIST=[".csv"])
    data = pd.DataFrame()
    data["path"] = file_paths
    data = data[~data['path'].apply(lambda x:os.path.split(x)[1].startswith('video_info'))].reset_index(drop=True)
    data["fold"] = (
        data["path"]
        .apply(lambda x: os.path.basename(x).split("_")[0])
        .astype(int)
        .astype(str)
    )
    data["subject_id"] = (
        data["path"]
        .apply(lambda x: os.path.basename(x).split("_")[1])
        .astype(int)
        .astype(str)
    )
    data = data.sort_values(["fold", "subject_id"]).reset_index(drop=True)

    columns_to_fold_by = data[
        [primary_column_name]
        + ([secondary_column_name] if secondary_column_name else [])
    ]

    train_val_test_file_paths = []
    unique_main_column_values = sorted(
        columns_to_fold_by[primary_column_name].unique()
    )

    if primary_column_name and secondary_column_name:
        for main_column_value in unique_main_column_values:
            corresponding_secondary_value = columns_to_fold_by[
                columns_to_fold_by[primary_column_name] == main_column_value
            ][secondary_column_name].unique()
            assert (
                len(corresponding_secondary_value) == 1
            ), f"Multiple Unique Secondary Values Found For Main Column Value {main_column_value} - {corresponding_secondary_value}"
            corresponding_secondary_value = corresponding_secondary_value[0]

            fold_train_val_test_file_paths = {"info": {}, "paths": {}}

            test_split = columns_to_fold_by[
                columns_to_fold_by[primary_column_name] == main_column_value
            ]
            fold_train_val_test_file_paths["paths"]["test"] = data.loc[
                test_split.index.to_list()
            ]["path"].to_list()
            fold_train_val_test_file_paths["info"]["test"] = {
                f"{primary_column_name}": sorted(
                    test_split[primary_column_name].unique().tolist()
                )
            }

            val_split = columns_to_fold_by[
                (columns_to_fold_by[primary_column_name] != main_column_value)
                & (
                    columns_to_fold_by[secondary_column_name]
                    == corresponding_secondary_value
                )
            ]
            fold_train_val_test_file_paths["paths"]["val"] = data.loc[
                val_split.index.to_list()
            ]["path"].to_list()
            fold_train_val_test_file_paths["info"]["val"] = {
                f"{primary_column_name}": sorted(
                    val_split[primary_column_name].unique().tolist()
                )
            }

            train_split = columns_to_fold_by[
                columns_to_fold_by[secondary_column_name]
                != corresponding_secondary_value
            ]
            fold_train_val_test_file_paths["paths"]["train"] = data.loc[
                train_split.index.to_list()
            ]["path"].to_list()
            fold_train_val_test_file_paths["info"]["train"] = {
                f"{primary_column_name}": sorted(
                    train_split[primary_column_name].unique().tolist()
                )
            }
            train_val_test_file_paths.append(fold_train_val_test_file_paths)

    elif primary_column_name:
        for main_column_value_i in unique_main_column_values:
            for main_column_value_j in unique_main_column_values:
                if main_column_value_i != main_column_value_j:
                    fold_train_val_test_file_paths = {"info": {}, "paths": {}}

                    test_split = columns_to_fold_by[
                        columns_to_fold_by[primary_column_name]
                        == main_column_value_i
                    ]
                    fold_train_val_test_file_paths["paths"]["test"] = data.loc[
                        test_split.index.to_list()
                    ]["path"].to_list()
                    fold_train_val_test_file_paths["info"]["test"] = {
                        f"{primary_column_name}": sorted(
                            test_split[primary_column_name].unique().tolist()
                        )
                    }

                    val_split = columns_to_fold_by[
                        columns_to_fold_by[primary_column_name]
                        == main_column_value_j
                    ]
                    fold_train_val_test_file_paths["paths"]["val"] = data.loc[
                        val_split.index.to_list()
                    ]["path"].to_list()
                    fold_train_val_test_file_paths["info"]["val"] = {
                        f"{primary_column_name}": sorted(
                            val_split[primary_column_name].unique().tolist()
                        )
                    }

                    train_split = columns_to_fold_by[
                        ~columns_to_fold_by[primary_column_name].isin(
                            [main_column_value_i, main_column_value_j]
                        )
                    ]
                    fold_train_val_test_file_paths["paths"]["train"] = data.loc[
                        train_split.index.to_list()
                    ]["path"].to_list()
                    fold_train_val_test_file_paths["info"]["train"] = {
                        f"{primary_column_name}": sorted(
                            train_split[primary_column_name].unique().tolist()
                        )
                    }

                    train_val_test_file_paths.append(fold_train_val_test_file_paths)

    return train_val_test_file_paths

In [4]:
input_directory_path = "/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed"
primary_column_name = 'fold'
secondary_column_name = None
train_val_test_file_paths = _get_train_val_test_file_paths(
    input_path = input_directory_path, 
    primary_column_name = primary_column_name, 
    secondary_column_name = secondary_column_name, 
)

In [5]:
train_val_test_file_paths[0]['paths']['train']

['/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha0.7_beta0/3_25_5_30fps_contrast_alpha0.7_beta0_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha0.7_beta0/3_25_10_30fps_contrast_alpha0.7_beta0_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha0.7_beta0/3_25_0_30fps_contrast_alpha0.7_beta0_preprocessed.csv',
 '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_pre

In [ ]:
import os
from tarfile import data_filter
from typing import Dict, List, Optional
from collections import Counter

import numpy as np
import pandas as pd
from keras.src.trainers.data_adapters.py_dataset_adapter import PyDataset
from src.preprocess.reshape import reshape_data_rnn
from src.utils.common import has_multiple_mappings, merge_intervals
from src.utils.load_data import _extract_headers, load_data


class MyDataGenerator(PyDataset):
    """
    Custom data generator for Keras.
    """

    def __init__(self, 
                 paths                  : List[str],
                 video_info_csv_path    : str,
                 mapping                : Dict,
                 groupby_column         : str = "file_name",
                 sortby_column          : str = "frame",
                 target_column          : str = "target",
                 columns_to_drop        : List[str] = [],
                 batch_size             : int = 32,
                 slice_size             : int = 3000,
                 step_size              : int = 3000,
                 discard_threshold_ratio: float = 0.6,
                 class_weights          : Optional[Dict] = {},
                 shuffle                : Optional[bool] = False,
                 **kwargs): 
        """
        Initialization function.  This is where you set up the basic parameters
        of your data generator.
        """
        super().__init__(**kwargs)

        self.paths                   = paths
        self.video_info_csv_path     = video_info_csv_path
        self.mapping                 = mapping
        self.groupby_column          = groupby_column
        self.sortby_column           = sortby_column
        self.target_column           = target_column
        self.columns_to_drop         = columns_to_drop
        self.batch_size              = batch_size
        self.slice_size              = slice_size
        self.step_size               = step_size
        self.discard_threshold_ratio = discard_threshold_ratio
        self.shuffle                 = shuffle
        
        self.columns_to_drop_1       = self._get_columns_to_drop_1()
        self.feature_columns         = self._get_feature_columns()
        self.batch_info              = self._get_batch_info(
            video_info_csv_path     = self.video_info_csv_path,
            batch_size              = self.batch_size,
            slice_size              = self.slice_size,
            step_size               = self.step_size,
            groupby_column          = 'path',
            groupby_column_filter   = self.paths,
            counts_column           = 'total_frames',
            discard_threshold_ratio = self.discard_threshold_ratio)
        self.true_labels             = self._get_true_labels()
        self.class_weights           = class_weights or self._calculate_class_weights()

        self.on_epoch_end()

    def _get_columns_to_drop_1(self):
        return [
            column_name
            for column_name in self.columns_to_drop
            if column_name not in [self.groupby_column, 
                                   self.sortby_column, 
                                   self.target_column]
        ]
    
    def _get_feature_columns(self):
        return [
            column_name
            for column_name in _extract_headers(self.paths[0])
            if column_name not in 
            self.columns_to_drop + [self.target_column, 
                                    self.sortby_column, 
                                    self.groupby_column]
        ]

    def _get_batch_info(
        self,
        video_info_csv_path    : str,
        batch_size             : int,
        slice_size             : int,
        step_size              : Optional[int],
        groupby_column         : str = 'path',
        groupby_column_filter  : List[str] = [],
        counts_column          : str = 'total_frames',
        discard_threshold_ratio: float = 0.6) -> Dict[str, List]: 

        info = {
            'counts' : [],
            'batches': [],
        }

        video_info = load_data(video_info_csv_path)
        groupby_column_filter = groupby_column_filter or video_info[groupby_column].unique()
        video_info = (
            video_info[video_info[groupby_column].isin(groupby_column_filter)]
            .assign(path=lambda x: pd.Categorical(x[groupby_column], categories=groupby_column_filter, ordered=True))
            .sort_values(groupby_column)
            .reset_index(drop=True)
        )
        
        new_rows = []
        for _, row in video_info.iterrows():
            total = row[counts_column]
            start_frame = 0
            while start_frame < total:
                end_frame = min(start_frame + slice_size, total)
                current_slice_size = end_frame - start_frame
                if current_slice_size >= discard_threshold_ratio * slice_size:
                    new_row = row.copy()  # Create a copy to avoid modifying the original row
                    new_row['start_frame'] = start_frame
                    new_row['end_frame'] = end_frame
                    new_rows.append(new_row)
                start_frame += step_size

        video_info_processed = pd.DataFrame(new_rows).reset_index(drop=True)

        for group_identifier, group in video_info_processed.groupby(groupby_column, sort=False):
            info['counts'].append({groupby_column : group_identifier, 
                                   'count' : len(group)})

        for i in range(int(np.ceil(len(video_info_processed)/batch_size))):
            start_index        = i * batch_size
            end_index          = min((i+1) * batch_size, len(video_info_processed))
            current_batch_info = []

            for group_identifier, group in video_info_processed[start_index: end_index].groupby(groupby_column, sort=False):
                current_batch_info.append(
                    {
                        groupby_column    : group_identifier,
                        'raw_intervals'   : list(zip(group['start_frame'], group['end_frame'])),
                        'merged_intervals': merge_intervals(list(zip(group['start_frame'], group['end_frame'])))
                    }
                )
            info['batches'].append(current_batch_info)

        return info
    
    def _get_true_labels(self):
        true_labels = {
            'g' : [],
            'y_true_unmapped' : [],
            'y' : []
        }

        for batch in self.batch_info['batches']:
            for file in batch:
                true_labels['g'].extend([os.path.basename(file['path'])] * len(file['raw_intervals']))
                true_labels['y_true_unmapped'].extend([os.path.basename(file['path']).split('_')[2]] * len(file['raw_intervals']))

        true_labels['y'] = [float(self.mapping[i]) for i in true_labels['y_true_unmapped']]

        assert has_multiple_mappings(true_labels['y_true_unmapped'], true_labels['g']) == False, 'Has Multiple Mapping'

        return true_labels

    def _calculate_class_weights(self):
        class_counts = Counter(self.true_labels['y'])
        total_samples = len(self.true_labels['y'])
        num_classes = len(class_counts)

        class_weights = {
            str(float(cls)): total_samples / (num_classes * count)
            for cls, count in class_counts.items()
        }

        return class_weights

    def on_epoch_end(self):
        """
        Called at the end of each epoch.
        #TODO implement proper shuffling. probably not needed as the batch indices are accessed without order
        """
        pass

    def __len__(self):
        """
        Returns the number of batches per epoch.  Keras uses this to know how many
        times to call `__getitem__` in one epoch.
        """
        assert len(self.batch_info['counts']) == len(self.paths), f"num_slices not calculated properly for all paths. num_slices : {len(self.batch_info['counts'])}, paths : {len(self.paths)}"

        return int(np.ceil(sum([file['count'] for file in self.batch_info['counts']])/self.batch_size))
    
    def _preprocess_data(self, file_data):
        file_data = file_data.drop(self.columns_to_drop_1, axis=1)
        X, y, g = reshape_data_rnn(df                      = file_data, 
                                   feature_columns         = self.feature_columns,
                                   slice_size              = self.slice_size,
                                   step_size               = self.step_size,
                                   groupby_column          = self.groupby_column,
                                   sortby_column           = self.sortby_column,
                                   target_column           = self.target_column,
                                   discard_threshold_ratio = self.discard_threshold_ratio)
        return X, y, g

    def __data_generation(self, index):
        """
        Generates data containing batch_size samples. This is where the actual data
        loading and preprocessing happens.
        """
        current_batch_info = self.batch_info['batches'][index]
        X, y, g, w = [], [], [], []

        for file in current_batch_info:
            file_data = load_data(input_path=file['path'], 
                      skiprows=file['merged_intervals'][0][0],
                      nrows=file['merged_intervals'][0][1] - file['merged_intervals'][0][0])
            X_file, y_file, g_file = self._preprocess_data(file_data=file_data)
            
            # Concatenate to the parent arrays
            X.extend(X_file)
            y.extend(y_file)
            g.extend(g_file)

        w = [self.class_weights[str(float(i))] for i in y]

        return {'X': np.array(X), 
                'y': np.array(y), 
                'g': np.array(g),
                'w': np.array(w)}

    def __getitem__(self, index):
        """
        Generates one batch of data.  The core of the data generator.
        #TODO Calculate class weights based on the name of files and num_slices and return
        """
        print(f"index {index}")
        batch = self.__data_generation(index)

        print(batch['y'].shape)
        print(batch['w'].shape)
        print(batch['X'].shape)
        
        return (batch['X'], batch['y'], batch['w'])

In [16]:
from keras.src.models.sequential import Sequential
import keras
from keras import layers

def _build_model(input_shape) -> Sequential:
    model = Sequential()
    model.add(layers.Input(shape=(input_shape[0], input_shape[1])))

    # Final output layer
    model.add(layers.Dense(1, activation="linear"))  # Use sigmoid activation

    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=0.001,
            weight_decay=0.00001,
            # beta_1=beta_1,
            # beta_2=beta_2,
            # epsilon=epsilon
        ),
        loss="mse",
        metrics=["mae"],
    )
    return model

In [17]:
model = _build_model((1000, 128))

In [18]:
input_directory_path = "/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed"
groupby_column, sortby_column, target_column = "file_name", "frame", "target"
video_info_csv_path = os.path.join(input_directory_path,'video_info_1.csv')
columns_to_drop = [
        "file_name",
        "file_name_1",
        "type",
        "frame",
        "fold",
        "subject_id",
    ]
batch_size              = 32
step_size               = 1000
slice_size              = 1000
discard_threshold_ratio = 0.5
mapping = {"10": 8.5, "5": 6.5, "0": 2}

train_data_gen = MyDataGenerator(
    paths                   = train_val_test_file_paths[0]['paths']['train'][:20],
    video_info_csv_path     = video_info_csv_path,
    groupby_column          = groupby_column,
    sortby_column           = sortby_column,
    target_column           = target_column,
    columns_to_drop         = columns_to_drop,
    batch_size              = batch_size,
    slice_size              = slice_size,
    step_size               = step_size,
    discard_threshold_ratio = discard_threshold_ratio,
    mapping                 = mapping,
    shuffle                 = True,
    workers                 = 1,
    use_multiprocessing     = False,
)
val_data_gen = MyDataGenerator(
    paths                   = train_val_test_file_paths[0]['paths']['val'][:20],
    video_info_csv_path     = video_info_csv_path,
    groupby_column          = groupby_column,
    sortby_column           = sortby_column,
    target_column           = target_column,
    columns_to_drop         = columns_to_drop,
    batch_size              = batch_size,
    slice_size              = slice_size,
    step_size               = step_size,
    discard_threshold_ratio = discard_threshold_ratio,
    mapping                 = mapping,
    shuffle                 = True,
    class_weights           = train_data_gen.class_weights,
    workers                 = 1,
    use_multiprocessing     = False,
)

2025-04-29 12:13:59,763 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/video_info_1.csv, data shape: (1575, 3)
2025-04-29 12:14:56,948 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/video_info_1.csv, data shape: (1575, 3)


In [11]:
a = val_data_gen.__getitem__(0)

index 0


2025-04-29 12:02:42,793 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/2_13_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18000, 135)
2025-04-29 12:02:44,274 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/2_13_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (14000, 135)


In [13]:
a[0].shape

(32, 1000, 128)

In [19]:
model.fit(train_data_gen, validation_data=val_data_gen, verbose=1)

index 0


2025-04-29 12:15:03,552 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18000, 135)
2025-04-29 12:15:05,057 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (14000, 135)


(32,)
(32,)
(32, 1000, 128)
index 1


2025-04-29 12:15:06,097 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (4000, 135)
2025-04-29 12:15:07,119 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/3_25_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18000, 135)
2025-04-29 12:15:08,725 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/contrast_alpha0.7_beta0/3_25_5_30fps_contrast_alpha0.7_beta0_preprocessed.csv, data shape: (10000, 135)


(32,)
(32,)
(32, 1000, 128)
index 8


...

(32,)
(32,)
(32, 1000, 128)
index 10


2025-04-29 12:15:13.704558: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Incompatible shapes: [32,1000] vs. [32]
	 [[{{function_node __inference_one_step_on_data_1487}}{{node gradient_tape/compile_loss/mse/mul/BroadcastGradientArgs}}]]


InvalidArgumentError: Graph execution error:

Detected at node gradient_tape/compile_loss/mse/mul/BroadcastGradientArgs defined at (most recent call last):
  File "/home/mojtaba.kananisarche/.conda/envs/.cvenv/lib/python3.9/runpy.py", line 197, in _run_module_as_main

  File "/home/mojtaba.kananisarche/.conda/envs/.cvenv/lib/python3.9/runpy.py", line 87, in _run_code

  File "/home/mojtaba.kananisarche/.conda/envs/.cvenv/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/mojtaba.kananisarche/.conda/envs/.cvenv/lib/python3.9/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/mojtaba.kananisarche/.conda/envs/.cvenv/lib/python3.9/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/mojtaba.kananisarche/.conda/envs/.cvenv/lib/python3.9/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/home/mojtaba.kananisarche/.conda/envs/.cvenv/lib/python3.9/asyncio/base_events.py", li

2025-04-29 12:15:14,518 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/grayscale/3_25_0_30fps_grayscale_preprocessed.csv, data shape: (4000, 135)
2025-04-29 12:15:15,742 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/rotate_angle5/3_25_10_30fps_rotate_angle5_preprocessed.csv, data shape: (18000, 135)
2025-04-29 12:15:17,372 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/rotate_angle5/3_25_0_30fps_rotate_angle5_preprocessed.csv, data shape: (10000, 135)


(32,)
(32,)
(32, 1000, 128)


In [ ]:
root_dir = "/work/long_lab/Mojtaba/data/UTA/1.3.3-augmented_embedded_preprocessed_normalized"
for subdir in os.listdir(root_dir):
    full_path = os.path.join(root_dir, subdir)
    if os.path.isdir(full_path):
        file_paths = get_file_paths(folder_path=full_path, 
                                    depth=2, 
                                    EXT_LIST=[".csv"])
                                    
        _ = get_video_info_from_csv(file_paths=file_paths, 
                                    output_directory_path=full_path, 
                                    output_file_prefix="video_info")

In [170]:
[os.path.basename(i) for i in sorted(f[0])] == [os.path.basename(i) for i in sorted(f[1])]

True

In [169]:
[os.path.basename(i) for i in sorted(f[1])]

['1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_01_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_01_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_02_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_02_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_02_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_03_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_03_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_03_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_04_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_04_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_04_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_05_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_05_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_05_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1_06_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
 '1

In [70]:
import os 

input_directory_path = "/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed"
video_info_csv_path = os.path.join(input_directory_path,'video_info_1.csv')

a, b = get_batch_info(
    video_info_csv_path = video_info_csv_path,
    batch_size = 32,
    slice_size= 1000,
    step_size= 1000,
    groupby_column = 'path',
    counts_column = 'total_frames',
    discard_threshold_ratio = 0.5)

2025-04-28 10:26:10,766 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/video_info_1.csv, data shape: (1575, 3)


In [71]:
b['batch']

[[{'path': '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_08_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
   'raw_intervals': [(0, 1000),
    (1000, 2000),
    (2000, 3000),
    (3000, 4000),
    (4000, 5000),
    (5000, 6000),
    (6000, 7000),
    (7000, 8000),
    (8000, 9000),
    (9000, 10000),
    (10000, 11000),
    (11000, 12000),
    (12000, 13000),
    (13000, 14000),
    (14000, 15000),
    (15000, 16000),
    (16000, 17000),
    (17000, 18000)],
   'merged_intervals': [(0, 18000)]},
  {'path': '/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_09_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv',
   'raw_intervals': [(0, 1000),
    (1000, 2000),
    (2000, 3000),
    (3000, 4000),
    (4000, 5000),
    (5000, 6000),
    (6000, 7000),
    (7000, 8000),
    (8000, 9000),
    (9000, 10000),
    (10000, 11000),
    (11000, 12000),
    (12000, 13000),
    (13000,

In [40]:
a[::32]

,file_name,path,total_frames,start_frame,end_frame
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,0,1000
32,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18362,14000,15000
64,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,19954,10000,11000


In [36]:
list(zip(i['start_frame'], i['end_frame']))

[(0, 1000),
 (1000, 2000),
 (2000, 3000),
 (3000, 4000),
 (4000, 5000),
 (5000, 6000),
 (6000, 7000),
 (7000, 8000),
 (8000, 9000),
 (9000, 10000),
 (10000, 11000),
 (11000, 12000),
 (12000, 13000),
 (13000, 14000)]

In [114]:
def merge_intervals(intervals):
    # First, sort the intervals by their start value
    intervals = sorted(intervals, key=lambda x: x[0])
    
    merged = []
    for interval in intervals:
        if not merged:
            merged.append(interval)
        else:
            last = merged[-1]
            if interval[0] <= last[1]:  # overlapping or touching
                merged[-1] = (last[0], max(last[1], interval[1]))  # merge
            else:
                merged.append(interval)
    return merged

# Example:
intervals = [(0, 10), (12, 15), (15, 25)]
print(merge_intervals(intervals))


[(0, 10), (12, 25)]


In [116]:
from src.utils.common import merge_intervals

ImportError: cannot import name 'merge_intervals' from 'src.utils.common' (/home/mojtaba.kananisarche/Zh/src/utils/common.py)

In [41]:
a.loc[0]

,file_name,path,total_frames,start_frame,end_frame
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,0,1000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,1000,2000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,2000,3000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,3000,4000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,4000,5000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,5000,6000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,6000,7000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,7000,8000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,8000,9000
0,video_info_1.csv,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156,9000,10000


In [154]:
dg.data

,X,y,g
0,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
1,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
2,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
3,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
4,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
5,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
6,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
7,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
8,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [147]:
dg.__getitem__(1)

2025-04-22 05:22:39,437 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_01_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18027, 135)


,X,y,g
7,"[[6.5, 1.418, 0.618, 0.09, -0.28, 0.878, 1.212...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
8,"[[6.5, 1.693, 1.159, -0.187, -0.224, 0.834, 1....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,"[[6.5, 0.975, 0.407, 0.227, -0.552, 1.425, 0.6...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
10,"[[6.5, 1.078, 0.301, -0.296, 0.095, 1.409, 0.7...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
11,"[[6.5, 1.022, 0.204, -0.076, -0.152, 0.835, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
12,"[[8.5, 0.723, 0.484, -0.139, 0.282, 1.075, 0.4...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
13,"[[8.5, 1.365, 0.639, -0.165, 0.032, 0.733, 0.5...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv


In [148]:
dg.data

,X,y,g
0,"[[2.0, 0.804, 0.005, 0.232, 0.04, 0.599, 1.208...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
1,"[[2.0, 0.975, 0.315, 0.186, 0.484, 1.249, 0.43...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
2,"[[2.0, -0.137, 0.268, 0.413, 0.714, 1.186, -0....",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
3,"[[2.0, 0.703, 0.149, 0.192, 0.053, 1.823, 0.55...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
4,"[[2.0, 0.237, 0.167, -0.129, 0.26, 1.162, 0.26...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
5,"[[2.0, 0.536, 0.584, 0.227, 0.304, 0.851, 0.31...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
6,"[[6.5, 1.412, 0.829, -0.153, -0.436, 1.186, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
7,"[[6.5, 1.418, 0.618, 0.09, -0.28, 0.878, 1.212...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
8,"[[6.5, 1.693, 1.159, -0.187, -0.224, 0.834, 1....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,"[[6.5, 0.975, 0.407, 0.227, -0.552, 1.425, 0.6...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [149]:
dg.stash

,X,y,g
14,"[[8.5, -0.284, -0.589, 0.133, -0.255, 2.441, -...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
15,"[[8.5, 1.099, -0.034, 0.45, 0.533, 1.466, 0.14...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
16,"[[8.5, 1.115, 0.46, 0.475, -0.145, 1.436, 0.11...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
17,"[[8.5, 1.565, 0.08, 0.382, 0.291, 1.759, 0.405...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv


In [33]:
for (X, y, g) in zip(a1, a2, a3):
    print(X, y, g)
    break

[[ 2.     0.804  0.005 ...  0.205 -1.507 -0.229]
 [ 2.     0.522 -0.369 ...  0.199 -1.104 -0.297]
 [ 2.     0.492 -0.019 ...  1.232 -0.433  0.135]
 ...
 [ 2.     0.619  0.678 ...  0.419 -1.322  0.369]
 [ 2.     0.562  0.629 ...  0.454 -1.341  0.35 ]
 [ 2.     0.792  0.627 ...  0.625 -1.295  0.273]] 2.0 1_01_0_30fps_brightness_alpha0.8_beta-10.csv


In [40]:
a2[0]

np.float64(2.0)

In [25]:
dg

NameError: name 'dg' is not defined

In [ ]:
keras.Sequential()
import inspect

print(inspect.getfile(Sequence))

/home/mojtaba.kananisarche/.conda/envs/.cvenv/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py


In [161]:
dg.stash.sample(len(dg.stash))

,X,y,g
8,"[[6.5, 1.693, 1.159, -0.187, -0.224, 0.834, 1....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,"[[6.5, 0.975, 0.407, 0.227, -0.552, 1.425, 0.6...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
10,"[[6.5, 1.078, 0.301, -0.296, 0.095, 1.409, 0.7...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
7,"[[6.5, 1.418, 0.618, 0.09, -0.28, 0.878, 1.212...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
11,"[[6.5, 1.022, 0.204, -0.076, -0.152, 0.835, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [ ]:
data = pd.read_csv(os.path.join(input_path, 'video_info_1.csv'))

In [73]:
popped, remaining = pop_row(data, slice(0,5))

In [74]:
popped

,path,total_frames
0,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156
1,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18362
2,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18519
3,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,19954
4,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,16820


In [75]:
remaining

,path,total_frames
5,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18222
6,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,15010
7,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18481
8,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18729
9,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18471
...,...,...
1570,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18206
1571,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,17645
1572,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18081
1573,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,20620


In [90]:
# %cd '/home/mojtaba.kananisarche/'

# from Zh.src.utils.load_data import load_data
# import numpy as np
# import pandas as pd
# from typing import List, Optional, Tuple, Optional
# from Zh.src.utils.file_folder import create_folders
# import os
# import json
# import pandas as pd
# import dataframe_image as dfi
# from Zh.src.utils.visualize_results import confusion_matrix_results, table_results

In [91]:
# path = "/home/mojtaba.kananisarche/Zh/outputs/2025-04-20|09:08:09|RNN_FACENET_5CV"

In [92]:
# data = table_results(
# results_directory_path  = path,
# output_directory_path   = None,
# output_file_path_prefix = 'results',
# cmap                    = "Blues",
# reverse_cmap            = False,
# header_1_bg_color       = 'silver',
# header_1_text_color     = 'black',
# header_2_bg_color       = 'darkgrey',
# header_2_text_color     = 'black',
# maximum_tries           = 1,
# slice_or_video          = 'video', #slice, video
# video_voting_method     = 'class_average', #majority, class_average, average
# height                  = 2,
# width                   = 5,
# )

In [93]:
# data = confusion_matrix_results(
#     results_directory_path  = path,
#     unique_classes          = [2, 6.5, 8.5],
#     output_directory_path   = None,
#     output_file_name_prefix = "results",
#     slice_or_video          = 'slice',     #slice, video
#     video_voting_method     = 'average',   #majority, class_average, average
#     index                   = -1,          # positive for any index and negative for all
#     split                   = "all",       # train, val, test, all
#     cmap                    = "Greys",
#     maximum_tries           = 5,
#     corner_text_color       = "white",
#     corner_bg_color         = "IndianRed",
#     pred_true_text_color    = "black",
#     pred_true_bg_color      = "silver",
#     labels_bg_color         = "beige",
#     labels_text_color       = "black",
#     height                  = 5,
#     width                   = None,
# )